In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import seaborn as sns
from sklearn.preprocessing import StandardScaler

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
#hyperparameter tuning and cross-validation
from sklearn.model_selection import GridSearchCV, StratifiedKFold
# model evaluation metrics

from sklearn.metrics import (
  classification_report, 
  confusion_matrix, 
  accuracy_score,
  precision_score,
  recall_score,
  f1_score,
  roc_auc_score,
  ConfusionMatrixDisplay,
  roc_curve)

In [137]:
dc = pd.read_csv('data/heartdisease_cleaned.csv')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_seq_items', None)
dc

,Age,Gender,Blood Pressure,Cholesterol Level,Exercise Habits,Smoking,Family Heart Disease,Diabetes,BMI,High Blood Pressure,Low HDL Cholesterol,High LDL Cholesterol,Alcohol Consumption,Stress Level,Sleep Hours,Sugar Consumption,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level,Heart Disease Status
0,56.0,Male,153.0,155.0,High,Yes,Yes,No,24.991591,Yes,Yes,No,High,Medium,7.633228,Medium,342.0,120.0,12.969246,12.387250,0
1,69.0,Female,146.0,286.0,High,No,Yes,Yes,25.221799,No,Yes,No,Medium,High,8.744034,Medium,133.0,157.0,9.355389,19.298875,0
2,46.0,Male,126.0,216.0,Low,No,No,No,29.855447,No,Yes,Yes,Low,Low,4.440440,Low,393.0,92.0,12.709873,11.230926,0
3,32.0,Female,122.0,293.0,High,Yes,Yes,No,24.130477,Yes,No,Yes,Low,High,5.249405,High,293.0,94.0,12.509046,5.961958,0
4,60.0,Male,166.0,242.0,Low,Yes,Yes,Yes,20.486289,Yes,No,No,Low,High,7.030971,High,263.0,154.0,10.381259,8.153887,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,25.0,Female,136.0,243.0,Medium,Yes,No,No,18.788791,Yes,No,Yes,Medium,High,6.834954,Medium,343.0,133.0,3.588814,19.132004,1
9996,38.0,Male,172.0,154.0,Medium,No,No,No,31.856801,Yes,No,Yes,Medium,High,8.247784,Low,377.0,83.0,2.658267,9.715709,1
9997,73.0,Male,152.0,201.0,High,Yes,No,Yes,26.899911,No,Yes,Yes,Medium,Low,4.436762,Low,248.0,88.0,4.408867,9.492429,1
9998,23.0,Male,142.0,299.0,Low,Yes,No,Yes,34.964026,Yes,No,Yes,Medium,High,8.526329,Medium,113.0,153.0,7.215634,11.873486,1


In [138]:
# label encoding the categorical features
import pickle 
encoders = {}

#apply label encoding and store the encoders in a dictionary for future use
for column in dc.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    dc[column] = le.fit_transform(dc[column])
    encoders[column] = le

# save the encoders to a pickle file for future use
with open("encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)
 # call the encoders
encoders

C:\Users\ayanr\AppData\Local\Temp\ipykernel_7368\1902946112.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in dc.select_dtypes(include=['object']).columns:


{'Gender': LabelEncoder(),
 'Exercise Habits': LabelEncoder(),
 'Smoking': LabelEncoder(),
 'Family Heart Disease': LabelEncoder(),
 'Diabetes': LabelEncoder(),
 'High Blood Pressure': LabelEncoder(),
 'Low HDL Cholesterol': LabelEncoder(),
 'High LDL Cholesterol': LabelEncoder(),
 'Alcohol Consumption': LabelEncoder(),
 'Stress Level': LabelEncoder(),
 'Sugar Consumption': LabelEncoder()}

Splitting the data set  \
Splitting the features and target

In [139]:
X = dc.drop('Heart Disease Status', axis=1)
Y = dc['Heart Disease Status']

In [140]:
print(X)

       Age  Gender  Blood Pressure  Cholesterol Level  Exercise Habits  \
0     56.0       1           153.0              155.0                0   
1     69.0       0           146.0              286.0                0   
2     46.0       1           126.0              216.0                1   
3     32.0       0           122.0              293.0                0   
4     60.0       1           166.0              242.0                1   
...    ...     ...             ...                ...              ...   
9995  25.0       0           136.0              243.0                2   
9996  38.0       1           172.0              154.0                2   
9997  73.0       1           152.0              201.0                0   
9998  23.0       1           142.0              299.0                1   
9999  38.0       0           128.0              193.0                2   

      Smoking  Family Heart Disease  Diabetes        BMI  High Blood Pressure  \
0           1                 

In [141]:
print(Y)

0       0
1       0
2       0
3       0
4       0
       ..
9995    1
9996    1
9997    1
9998    1
9999    1
Name: Heart Disease Status, Length: 10000, dtype: int64


Splitting into training and test data 

In [142]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)
#note stratify=Y ensures that your training and testing datasets maintain approximately 
# the same class distribution as your original target variable (Y).

In [143]:
print(X.shape, X_train.shape, X_test.shape)

(10000, 20) (8000, 20) (2000, 20)


In [144]:
print(Y.shape, Y_train.shape, Y_test.shape)

(10000,) (8000,) (2000,)


Checking the target distribution after splitting \
Before scaling or training, let's confirm that the class distribution was maintained.

In [145]:
# check the distribution of the target variable in the training data.

print("Training target distribution:")
print(Y_train.value_counts())

#check the distribution of the target variable in the testing data.
print("\nTesting target distribution:")
print(Y_test.value_counts())

# check the percentages in the training data.
print("\n Training target percentages:")
print(Y_train.value_counts(normalize=True)*100)

# check the percentages in the testing data.
print("\n Testing target percentages:")
print(Y_test.value_counts(normalize=True)*100)


Training target distribution:
Heart Disease Status
0    6400
1    1600
Name: count, dtype: int64

Testing target distribution:
Heart Disease Status
0    1600
1     400
Name: count, dtype: int64

 Training target percentages:
Heart Disease Status
0    80.0
1    20.0
Name: proportion, dtype: float64

 Testing target percentages:
Heart Disease Status
0    80.0
1    20.0
Name: proportion, dtype: float64


Feature Scaling Stage \
Using Standard Scaler 

In [146]:
# create the StandardScalar object
scaler = StandardScaler()

#learning the scaling information from the training data and scale it
X_train_scaled = scaler.fit_transform(X_train)

#Apply the same scaling information to the testing data
X_test_scaled = scaler.transform(X_test)

In [147]:
print(X_train_scaled)

[[-1.6623689   1.00050013  1.43693945 ...  1.64319913 -0.38070827
   1.28023824]
 [-1.49751064 -0.99950012 -1.51522868 ...  0.07496781 -0.98541248
  -1.09904334]
 [-0.23359729 -0.99950012  0.47180756 ... -0.47603238  1.34524192
   1.69001315]
 ...
 [-0.23359729  1.00050013 -0.55009679 ... -1.0694172  -0.92779882
  -1.01439057]
 [-0.50836106 -0.99950012  1.60725684 ...  0.49881411 -0.47943848
  -0.79729895]
 [ 0.42583576 -0.99950012 -1.57200114 ... -0.73034016 -0.94675408
   1.50339946]]


In [148]:
print(X_test_scaled)

[[ 0.75555229  1.00050013 -1.11782143 ... -0.00980145 -0.16423293
   0.72265163]
 [ 0.20602475  1.00050013 -1.57200114 ... -0.17933997  1.09114397
   1.01735306]
 [-1.60741615  1.00050013 -0.60686925 ... -1.02703257  0.62988121
  -0.41754091]
 ...
 [-1.16779411 -0.99950012 -0.77718664 ...  0.625968   -0.21761856
  -1.35444258]
 [-0.83807759 -0.99950012 -1.0042765  ... -0.60318627  1.56897047
   0.69267949]
 [ 0.20602475  1.00050013 -0.83395911 ... -0.26410923 -0.74052337
  -0.08474248]]


Lets inspect the Scaled trained data but in a Dataframe

In [149]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)

#display the first five rows of the sclaed training data
X_train_scaled_df.head()

,Age,Gender,Blood Pressure,Cholesterol Level,Exercise Habits,Smoking,Family Heart Disease,Diabetes,BMI,High Blood Pressure,Low HDL Cholesterol,High LDL Cholesterol,Alcohol Consumption,Stress Level,Sleep Hours,Sugar Consumption,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level
7395,-1.662369,1.0005,1.436939,1.065133,-1.212502,-1.034080,1.003506,-1.002503,-0.044409,0.986590,1.009546,-0.988813,-0.325153,-0.007197,1.341135,-1.227598,-0.576281,1.643199,-0.380708,1.280238
5642,-1.497511,-0.9995,-1.515229,-1.279291,-1.212502,-1.034080,-0.996506,-1.002503,0.173957,-1.013592,1.009546,1.011314,0.887539,-1.232242,0.006903,0.005085,0.961310,0.074968,-0.985412,-1.099043
8866,-0.233597,-0.9995,0.471808,-1.233322,1.220714,0.967043,1.003506,0.997503,0.762484,0.986590,-0.990545,1.011314,0.887539,1.217847,1.118184,0.005085,1.443241,-0.476032,1.345242,1.690013
8517,-1.112841,1.0005,-0.379779,1.524824,0.004106,0.967043,-0.996506,0.997503,-1.204844,-1.013592,1.009546,-0.988813,-1.537846,-1.232242,1.635598,1.237768,1.362920,-0.009801,-1.567248,0.073432
3706,-0.178645,-0.9995,-0.833959,-0.038126,0.004106,-1.034080,1.003506,-1.002503,-0.140953,-1.013592,-0.990545,1.011314,-1.537846,-1.232242,0.556967,-1.227598,1.144903,1.219353,0.793701,1.333975


In [150]:
X_train_scaled_df.describe()



,Age,Gender,Blood Pressure,Cholesterol Level,Exercise Habits,Smoking,Family Heart Disease,Diabetes,BMI,High Blood Pressure,Low HDL Cholesterol,High LDL Cholesterol,Alcohol Consumption,Stress Level,Sleep Hours,Sugar Consumption,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level
count,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03
mean,-3.108624e-17,-1.154632e-16,-5.400125e-16,5.329071e-17,7.815970e-17,-3.219647e-17,-1.705303e-16,6.217249e-17,5.409007e-16,-9.681145e-17,1.065814e-16,-3.552714e-18,5.861978e-17,-1.225686e-16,-6.039613e-17,7.904788e-17,4.263256e-17,-1.367795e-16,-2.975398e-17,-6.128431e-17
std,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00
min,-1.717322e+00,-9.995001e-01,-1.685546e+00,-1.738982e+00,-1.212502e+00,-1.034080e+00,-9.965061e-01,-1.002503e+00,-1.770164e+00,-1.013592e+00,-9.905447e-01,-9.888126e-01,-1.537846e+00,-1.232242e+00,-1.710162e+00,-1.227598e+00,-1.735212e+00,-1.705187e+00,-1.726188e+00,-1.713853e+00
25%,-8.380776e-01,-9.995001e-01,-8.907316e-01,-8.655694e-01,-1.212502e+00,-1.034080e+00,-9.965061e-01,-1.002503e+00,-8.508187e-01,-1.013592e+00,-9.905447e-01,-9.888126e-01,-3.251532e-01,-1.232242e+00,-8.753806e-01,-1.227598e+00,-8.631451e-01,-8.998787e-01,-8.762377e-01,-8.627243e-01
50%,-1.378627e-02,-9.995001e-01,1.762785e-02,7.843478e-03,4.106053e-03,9.670428e-01,-9.965061e-01,9.975031e-01,3.903284e-03,9.865899e-01,-9.905447e-01,-9.888126e-01,8.875393e-01,-7.197137e-03,6.735350e-03,5.084818e-03,-2.553090e-03,-9.801446e-03,-3.582562e-04,-6.866850e-03
75%,8.654578e-01,1.000500e+00,8.692148e-01,8.582718e-01,1.220714e+00,9.670428e-01,1.003506e+00,9.975031e-01,8.666542e-01,9.865899e-01,1.009546e+00,1.011314e+00,8.875393e-01,1.217847e+00,8.744118e-01,1.237768e+00,8.695135e-01,8.802758e-01,8.672078e-01,8.475169e-01
max,1.689749e+00,1.000500e+00,1.720802e+00,1.708700e+00,1.220714e+00,9.670428e-01,1.003506e+00,9.975031e-01,1.719462e+00,9.865899e-01,1.009546e+00,1.011314e+00,8.875393e-01,1.217847e+00,1.720567e+00,1.237768e+00,1.707156e+00,1.685584e+00,1.738505e+00,1.749851e+00


In [151]:
# check the mean of each scaled feature.
print(X_train_scaled_df.mean())

Age                    -3.108624e-17
Gender                 -1.154632e-16
Blood Pressure         -5.400125e-16
Cholesterol Level       5.329071e-17
Exercise Habits         7.815970e-17
Smoking                -3.219647e-17
Family Heart Disease   -1.705303e-16
Diabetes                6.217249e-17
BMI                     5.409007e-16
High Blood Pressure    -9.681145e-17
Low HDL Cholesterol     1.065814e-16
High LDL Cholesterol   -3.552714e-18
Alcohol Consumption     5.861978e-17
Stress Level           -1.225686e-16
Sleep Hours            -6.039613e-17
Sugar Consumption       7.904788e-17
Triglyceride Level      4.263256e-17
Fasting Blood Sugar    -1.367795e-16
CRP Level              -2.975398e-17
Homocysteine Level     -6.128431e-17
dtype: float64


In [152]:
# check the standard deviation of each scaled feature.
print(X_train_scaled_df.std().round(4))

Age                     1.0001
Gender                  1.0001
Blood Pressure          1.0001
Cholesterol Level       1.0001
Exercise Habits         1.0001
Smoking                 1.0001
Family Heart Disease    1.0001
Diabetes                1.0001
BMI                     1.0001
High Blood Pressure     1.0001
Low HDL Cholesterol     1.0001
High LDL Cholesterol    1.0001
Alcohol Consumption     1.0001
Stress Level            1.0001
Sleep Hours             1.0001
Sugar Consumption       1.0001
Triglyceride Level      1.0001
Fasting Blood Sugar     1.0001
CRP Level               1.0001
Homocysteine Level      1.0001
dtype: float64


Model Training stage \
 Using Logistic Regrression

In [153]:
model = LogisticRegression(
  class_weight="balanced",
  max_iter=1000,
  random_state=42
)

# train the model using the training dataset.
model.fit(X_train_scaled, Y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to u

Before hyperparameter tuning, lets make baseline predictions

In [154]:
# predict the class labels for the testing dataset.
Y_pred = model.predict(X_test_scaled)

# predict the probability of class 1 (heart disease).
Y_prob = model.predict_proba(X_test_scaled)[:, 1]

In [159]:
print(pd.Series(Y_pred).value_counts)

<bound method IndexOpsMixin.value_counts of 0       1
1       0
2       0
3       0
4       0
       ..
1995    0
1996    0
1997    1
1998    1
1999    0
Length: 2000, dtype: int64>


In [156]:
print(Y_prob)

[0.52327306 0.47482614 0.46969245 ... 0.51653303 0.53145022 0.42293382]


define the evaluation metrics

In [157]:
# calculate baseline evaluation metrics.
baseline_accuracy = accuracy_score(Y_test, Y_pred)

baseline_precision = precision_score(
  Y_test, Y_pred, zero_division=0
)

baseline_recall = recall_score(
  Y_test, Y_pred, zero_division=0
)

baseline_f1 = f1_score(
    Y_test, Y_pred, zero_division=0
)

baseline_auc = roc_auc_score(Y_test, Y_prob)

# Display the results.
print("BASELINE LOGISTIC REGRESSION")
print("----------------------------")
print("Accuracy:", round(baseline_accuracy, 4))
print("Precision:", round(baseline_precision, 4))
print("Recall:", round(baseline_recall, 4))
print("F1-score:", round(baseline_f1, 4))
print("ROC-AUC:", round(baseline_auc, 4))

BASELINE LOGISTIC REGRESSION
----------------------------
Accuracy: 0.5075
Precision: 0.1905
Recall: 0.45
F1-score: 0.2677
ROC-AUC: 0.4861


In [160]:
print(X_train_scaled.shape)
print(X_test_scaled.shape)

(8000, 20)
(2000, 20)


In [164]:
# Define the hyperparameter combinations for the balanced model.
from sklearn.model_selection import StratifiedKFold


# Create a 5-fold stratified cross-validation strategy.
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


balanced_param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs']
}

# Create the balanced Logistic Regression model.
balanced_model = LogisticRegression(
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

# Set up GridSearchCV for the balanced model.
balanced_grid_search = GridSearchCV(
    estimator=balanced_model,
    param_grid=balanced_param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    refit=True
)

# Train and search for the best hyperparameters.
balanced_grid_search.fit(X_train_scaled, Y_train)

# Display the best parameters.
print("Best Balanced Parameters:", balanced_grid_search.best_params_)

# Display the best cross-validation ROC-AUC.
print(
    "Best Balanced Cross-validation ROC-AUC:",
    balanced_grid_search.best_score_
)

Best Balanced Parameters: {'C': 0.001, 'solver': 'lbfgs'}
Best Balanced Cross-validation ROC-AUC: 0.5082172851562499


In [165]:
# Retrieve the best balanced Logistic Regression model.
best_balanced_model = balanced_grid_search.best_estimator_

# Predict the class labels.
Y_pred_balanced = best_balanced_model.predict(X_test_scaled)

# Predict the probability of heart disease.
Y_prob_balanced = best_balanced_model.predict_proba(X_test_scaled)[:, 1]